# Systematics: what the blocks look like

The sampler's model is `d = w * (Us s + Uf f) + n`. `imgibbs.systematics` adds
an optional third component,

$$d = w\,(U_s s + U_f f + U_g g) + n$$

where $U_g$ is a **fixed, low-rank** basis of systematic templates and $g$ a
short vector of amplitudes sampled jointly with everything else.

Three are implemented: **ground spill** (§1-6), **polarisation leakage** (§7)
and **1/f noise** (§8). §9 puts them side by side.

This notebook shows the *structure* of those blocks — what the templates are,
why they were chosen, and what a foreground clean does to them. It does not
run the sampler. For that see `scripts/systematics_injection.py` and
`docs/STATUS.md`.

Everything here runs on data shipped with the repository; the MeerKLASS L2021
cubes are not needed.


In [ ]:
# Run from the repo without installing: `pip install -e ..` makes this unnecessary.
import sys, pathlib
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

from imgibbs import (
    RM_DEFAULT, data_path, faraday_templates, groundspill_basis,
    groundspill_cube, lambda_squared, leakage_basis, load, onef_basis,
    period_scan, ripple_wavenumber, scan_templates, spectral_templates,
    spillover_envelope, survey_grid,
)
import json

# The live grid. Everything follows CROP -- see imgibbs/grid.py.
CROP = (slice(33, 103), slice(14, 59), slice(0, 250))
SHAPE = (70, 45, 250)
N_MODES = 6          # foreground modes the sampler actually runs with
PERIOD = 17.5        # standing-wave period, MHz

grid = survey_grid(CROP, SHAPE)
FREQS = grid.freqs
BAND = FREQS[-1] - FREQS[0]
print(grid.summary())
print(f'\nband width   : {BAND:.2f} MHz')

In [ ]:
# Plot style: recessive chrome, categorical hues assigned in fixed order.
BLUE, ORANGE, AQUA, YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID, AXIS = '#e1e0d9', '#c3c2b7'

plt.rcParams.update({
    'figure.dpi': 140, 'savefig.dpi': 140,
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': AXIS, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.linewidth': 0.8, 'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': GRID, 'grid.linewidth': 0.6,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'xtick.labelcolor': INK2, 'ytick.labelcolor': INK2,
    'legend.frameon': False, 'font.size': 9, 'lines.linewidth': 1.8,
})

# Diverging: two poles + a NEUTRAL GRAY midpoint, for signed patterns.
DIVERGING = LinearSegmentedColormap.from_list(
    'gs_div', ['#0d366b', '#2a78d6', '#f0efec', '#e34948', '#8f2320'])


def bare(ax, right=True, top=True):
    """Drop the spines that carry no information."""
    for side in (['right'] if right else []) + (['top'] if top else []):
        ax.spines[side].set_visible(False)


def signed(ax, field, label=None, lim=None, transpose=True):
    """Show a signed map on the diverging scale, centred on zero.

    Pass ``lim`` to share one scale across panels. Without it each panel is
    normalised to its own maximum, which makes a CONSTANT pattern saturate and
    read as extreme when it is merely uniform.
    """
    lim = lim or (np.abs(field).max() or 1.0)
    im = ax.imshow(field.T if transpose else field, origin='lower',
                   cmap=DIVERGING, aspect='auto',
                   norm=TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim))
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if label:
        ax.set_title(label, fontsize=9, color=INK)
    return im

## 1. The spectrum: a smooth envelope and a ripple

Ground spill is far-sidelobe pickup of ~280 K ground emission. Two pieces, and
they behave completely differently in this sampler:

* a **smooth envelope** — the spillover fraction rises toward low frequency
  roughly as the beam solid angle, $(\nu/\nu_{\rm ref})^{-2}$;
* a **standing-wave ripple**, set by the round trip between dish and
  subreflector, modulating that envelope.

The ripple enters the basis as a **cosine/sine pair** rather than an amplitude
and a phase. Amplitude and phase are a nonlinear pair; the two quadratures are
linear parameters, which is the whole reason this block fits inside the
constrained realisation with no Metropolis step. The *period* is genuinely
nonlinear, and is fixed rather than sampled.

In [ ]:
env = spillover_envelope(FREQS)
templates = spectral_templates(FREQS, period=PERIOD, n_harmonics=1)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(7.2, 4.4), sharex=True,
                               gridspec_kw={'height_ratios': [1, 1.3]})

ax0.plot(FREQS, env, color=BLUE)
ax0.set_ylabel('relative spill')
ax0.set_title('Smooth spillover envelope  $(\\nu/\\nu_{\\rm ref})^{-2}$',
              loc='left')
bare(ax0)

ax1.plot(FREQS, templates[0], color=ORANGE, label='cosine quadrature')
ax1.plot(FREQS, templates[1], color=AQUA, label='sine quadrature')
ax1.set_xlabel('frequency [MHz]')
ax1.set_ylabel('unit-RMS template')
ax1.set_title(f'Ripple quadratures, {PERIOD:g} MHz period '
              f'({BAND / PERIOD:.2f} cycles across the band)', loc='left')
ax1.legend(loc='upper center', ncols=2, bbox_to_anchor=(0.5, -0.3))
bare(ax1)

fig.tight_layout()

## 2. What the foreground block can already absorb

This is the measurement the whole design rests on.

The foreground block has **per-pixel free amplitudes** on `n_modes` smooth
frequency modes. So anything smooth in frequency lies inside its span *whatever
its spatial structure is* — there is no spatial pattern distinctive enough to
rescue it.

Two consequences:

* The **smooth envelope is absorbed to machine precision** at every `n_modes`.
  It is unidentifiable, and harmless for exactly the same reason. That is why
  `spectral_templates` leaves it out by default: putting it in $U_g$ would only
  add a direction the data cannot constrain.
* The **ripple survives**, and how much survives depends on the period *and*
  the bandwidth. Raising `n_modes` does absorb it — but `docs/STATUS.md`
  records that `n_modes = 20` removes the 21cm signal too ($T(k)$ 0.006–0.06)
  and pushes $\tau_{\rm int}$ to 54–105. That cure is worse than the disease.

The lower panel is why it matters: a ripple is a **single $k_\parallel$ mode**,
so it does not spread across bins — it dumps all of its power into one.

In [ ]:
def legendre_basis(n_freq, n_modes):
    """The sampler's foreground basis: orthonormalised Legendre polynomials."""
    vander = np.polynomial.legendre.legvander(np.linspace(-1, 1, n_freq),
                                              n_modes - 1)
    return np.linalg.qr(vander)[0].T


periods = np.logspace(np.log10(2.0), np.log10(60.0), 160)
k_of_period = ripple_wavenumber(periods, BAND, grid.box_dims[2])

# The sampler's ACTUAL k-bins, recovered from the shipped S rather than
# re-derived. kbins_from_crop takes k_min from the observed footprint, so
# calling it on anything but the real mask silently gives different bins --
# which is the trap docs/STATUS.md describes at length. S is piecewise-constant
# over the bins, so its distinct values ARE the binning, and the counts check
# against the metadata the run recorded.
meta = json.loads(data_path('S_starting_point_cropped_meta.json').read_text())
sig_k = np.array(meta['kbin_sig_k'])
_S = load('S_starting_point_cropped.npy')
_vals, _counts = np.unique(_S, return_counts=True)

idxs = np.zeros(_S.size, dtype=int)
for i, n_modes_in_bin in enumerate(meta['kbin_modes_per_bin']):
    match = _vals[_counts == n_modes_in_bin]
    assert len(match) == 1, f'bin {i}: {len(match)} S values with {n_modes_in_bin} modes'
    idxs[_S == match[0]] = i + 1
assert (idxs > 0).sum() == sum(meta['kbin_modes_per_bin'])
kbin_meta = {'k_min': meta['kbin_k_min'], 'k_max': meta['kbin_k_max']}
print('k-bins       : ' + ', '.join(f'{k:.4f}' for k in sig_k))
print('modes/bin    : ' + str(meta['kbin_modes_per_bin']))

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(7.2, 6.0), sharex=True,
                               gridspec_kw={'height_ratios': [1.25, 1]})

for n_modes, colour in ((6, BLUE), (10, ORANGE), (20, AQUA)):
    absorbed = period_scan(FREQS, periods, legendre_basis(len(FREQS), n_modes))
    ax0.plot(periods, absorbed, color=colour, label=f'{n_modes} modes')
    # Label each curve where they are furthest apart -- their 50% crossings --
    # rather than at the right edge, where all three sit on top of each other.
    ax0.annotate(f'{n_modes} modes', xy=(np.interp(0.5, absorbed, periods), 0.5),
                 xytext=(9, -20), textcoords='offset points',
                 ha='left', fontsize=8.5, color=colour)

env_absorbed = float(np.sum((legendre_basis(len(FREQS), 6)
                             @ (env / np.linalg.norm(env))) ** 2))
ax0.axhline(env_absorbed, color=MUTED, lw=1.0, ls=(0, (4, 3)))
ax0.annotate(f'the smooth envelope sits here: {env_absorbed:.4f}, '
             f'at every n_modes', xy=(2.2, env_absorbed), xytext=(0, 5),
             textcoords='offset points', fontsize=8.5, color=INK2)

ax0.set_ylabel('fraction absorbed by the foreground')
ax0.set_ylim(-0.04, 1.16)
ax0.set_title('What a foreground clean already removes', loc='left')
ax0.legend(loc='center left', fontsize=8.5)
bare(ax0)

# Where the ripple lands in k, against the sampler's actual bins.
edges = np.concatenate([[kbin_meta['k_min']], np.sqrt(sig_k[1:] * sig_k[:-1]),
                        [kbin_meta['k_max']]])
for i, k in enumerate(sig_k):
    ax1.axhspan(edges[i], edges[i + 1], color=GRID,
                alpha=0.55 if i % 2 == 0 else 0.25, lw=0)
    # Right-aligned at the right edge: the curve falls left-to-right, so that
    # corner is the one part of the panel it never crosses.
    ax1.annotate(f'bin {i}   k = {k:.3f}', xy=(periods[-1], k), xytext=(-4, 0),
                 textcoords='offset points', fontsize=8, color=INK2,
                 ha='right', va='center')

ax1.plot(periods, k_of_period, color=BLUE)
ax1.set_yscale('log')
ax1.set_ylim(k_of_period.min() * 0.75, float(kbin_meta['k_max']) * 1.15)
ax1.set_xscale('log')
ax1.set_xlabel('ripple period [MHz]')
ax1.set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
ax1.set_title(r'Where that ripple lands: a single $k_\parallel$ mode,'
              ' all in one bin', loc='left')
bare(ax1)

for ax in (ax0, ax1):
    ax.axvline(PERIOD, color=INK2, lw=1.0, ls=(0, (2, 2)))
ax1.annotate(f'{PERIOD:g} MHz default', xy=(PERIOD, kbin_meta['k_max']),
             xytext=(-5, -4), textcoords='offset points', fontsize=8.5,
             color=INK2, ha='right', va='top')
ax1.set_xticks([2, 5, 10, 20, 40])
ax1.set_xticklabels(['2', '5', '10', '20', '40'])

fig.tight_layout()

k_hit = ripple_wavenumber(PERIOD, BAND, grid.box_dims[2])
print(f'{PERIOD:g} MHz ripple -> k_par {k_hit:.4f} Mpc^-1, '
      f'nearest bin {int(np.argmin(np.abs(sig_k - k_hit)))}')
print('foreground absorbs',
      ', '.join(f'n={n}: {period_scan(FREQS, [PERIOD], legendre_basis(len(FREQS), n))[0]:.1%}'
                for n in (6, 10, 20)))

## 3. The spatial side: low order, not per-pixel

A drift scan at fixed elevation sees a ground pattern that is **constant in the
telescope frame**, so to lowest order it contributes the same in every map
pixel. What breaks that degeneracy is the azimuth covered within a scan, which
maps onto the scan direction of the map.

So the spatial model is a low-order polynomial along the scan axis — constant
plus gradient by default. Deliberately **not** per-pixel: per-pixel amplitudes
on a frequency template is exactly what $U_f$ already is, so a systematic with
that much spatial freedom would be unidentifiable no matter how distinctive its
spectrum.

In [ ]:
spatial = scan_templates(SHAPE, order=1, scan_axis=0)
names = ['constant\n(the pure drift-scan limit)', 'scan-direction gradient']

LIM = np.abs(spatial).max()      # one scale for both panels

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6))
for ax, row, name in zip(axes, spatial, names):
    im = signed(ax, row.reshape(SHAPE[0], SHAPE[1]), name, lim=LIM)
    ax.set_xlabel('RA pixel (scan direction)', fontsize=8)
axes[0].set_ylabel('Dec pixel', fontsize=8)
fig.colorbar(im, ax=axes, label='unit-RMS amplitude', pad=0.02, shrink=0.9)

## 4. The basis is separable, and that is what makes it cheap

Each template is an outer product, `spatial[s] x spectral[t]`, so the amplitude
array `g` has shape `(n_s, n_t)` — **four numbers by default**.

Separability is not a cosmetic choice. It means:

* applying $U_g$ is two small matrix multiplies rather than a pass over
  ~800,000 voxels;
* the Gram matrix the preconditioner needs **factorises exactly**,
  $U_g^{\mathsf T}U_g = \mathrm{kron}(G_{\rm spatial}, G_{\rm spectral})$ — an
  identity, not an approximation.

The grid below is literally that outer product: rows are spatial patterns,
columns are spectral quadratures, and each panel is one basis element shown at
a single frequency channel.

In [ ]:
basis = groundspill_basis(FREQS, SHAPE, period=PERIOD, order=1)
print(f'{basis.n_s} spatial x {basis.n_t} spectral = {basis.n_params} '
      f'parameters, g has shape {basis.g_shape}')

# Each product is shown as an RA-frequency slice at mid-Dec, NOT a sky map at
# one channel: a single channel cannot show the difference between the two
# quadratures, because they differ in phase ALONG frequency. In this view the
# phase shift between the columns is the visible thing.
dec = SHAPE[1] // 2

fig = plt.figure(figsize=(7.4, 4.5))
gs = fig.add_gridspec(3, 3, height_ratios=[0.5, 1, 1],
                      width_ratios=[0.45, 1, 1], hspace=0.28, wspace=0.16)

# Top row: the two spectral templates.
for t in range(basis.n_t):
    ax = fig.add_subplot(gs[0, t + 1])
    ax.plot(FREQS, basis.spectral[t], color=ORANGE if t == 0 else AQUA, lw=1.3)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for side in ('top', 'right', 'left', 'bottom'):
        ax.spines[side].set_visible(False)
    ax.set_title(f'spectral {t}  ({"cos" if t == 0 else "sin"})', fontsize=8.5,
                 color=INK2)

# One scale across every map below, so a uniform pattern does not saturate.
LIM = max(np.abs(basis.spatial).max(),
          max(np.abs(basis.cube(np.eye(basis.n_params)[i])[:, dec, :]).max()
              for i in range(basis.n_params)))

# Left column: the two spatial patterns, as sky maps.
for sp in range(basis.n_s):
    ax = fig.add_subplot(gs[sp + 1, 0])
    signed(ax, basis.spatial[sp].reshape(SHAPE[0], SHAPE[1]), lim=LIM)
    ax.set_ylabel(f'spatial {sp}\n({"constant" if sp == 0 else "gradient"})',
                  fontsize=8.5, color=INK2)

# The products, as RA x frequency.
for sp in range(basis.n_s):
    for t in range(basis.n_t):
        ax = fig.add_subplot(gs[sp + 1, t + 1])
        g = np.zeros(basis.g_shape)
        g[sp, t] = 1.0
        im = signed(ax, basis.cube(g)[:, dec, :], lim=LIM, transpose=False)
        if sp == basis.n_s - 1:
            ax.set_xlabel('frequency', fontsize=8, color=MUTED)
        if t == 0 and sp == 0:
            ax.set_ylabel('RA', fontsize=8, color=MUTED)
fig.colorbar(im, ax=fig.axes, label='amplitude', pad=0.015, shrink=0.55,
             aspect=18)

fig.suptitle(f'The four basis elements: spatial pattern x spectral template',
             x=0.02, ha='left', fontsize=10, color=INK)

## 5. What a foreground clean leaves behind

Inject ground spill with a large smooth part and a small ripple, then fit and
subtract the same 6-mode Legendre basis the sampler uses.

The smooth part is 500x the ripple on the way in, and **gone** on the way out.
The ripple is what remains — which is the entire argument for modelling it
separately rather than buying the foreground more polynomials.

In [ ]:
RIPPLE_RMS, SPILL_LEVEL = 1e-3, 0.5
spill, truth = groundspill_cube(FREQS, SHAPE, ripple_rms=RIPPLE_RMS,
                                spill_level=SPILL_LEVEL, period=PERIOD,
                                rng=np.random.default_rng(0))
ripple_only = truth['basis'].cube(truth['g_true'])

fg = legendre_basis(len(FREQS), N_MODES)
residual = spill.reshape(-1, len(FREQS))
residual = (residual - (residual @ fg.T) @ fg).reshape(SHAPE)

px = (SHAPE[0] // 3, SHAPE[1] // 2)
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(7.2, 4.6), sharex=True)

ax0.plot(FREQS, spill[px], color=BLUE)
ax0.set_ylabel('T [K]')
ax0.set_title(f'Injected ground spill in one pixel  '
              f'(smooth {SPILL_LEVEL:g} K + ripple {RIPPLE_RMS:g} K)',
              loc='left')
bare(ax0)

ax1.plot(FREQS, residual[px] * 1e3, color=ORANGE,
         label=f'after a {N_MODES}-mode Legendre clean')
ax1.plot(FREQS, ripple_only[px] * 1e3, color=INK2, lw=1.1, ls=(0, (4, 3)),
         label='the injected ripple')
ax1.set_xlabel('frequency [MHz]')
ax1.set_ylabel('T [mK]')
ax1.set_title('What survives the clean — note the axis is now mK', loc='left')
ax1.legend(loc='upper center', ncols=2, bbox_to_anchor=(0.5, -0.28))
bare(ax1)

fig.tight_layout()

print(f'injected  : smooth {SPILL_LEVEL:.3g} K, ripple {RIPPLE_RMS:.3g} K')
print(f'residual  : {residual.std():.3e} K after the clean')
print(f'            = {residual.std() / RIPPLE_RMS:.1%} of the injected ripple,'
      f' and {residual.std() / SPILL_LEVEL:.2e} of the smooth part')

## 6. Why it matters: the residual sits on the signal bins

The simulated H I cube is the truth curve the sampler is judged against. The
ripple residual lands on top of it, in one bin.

The contamination is **not broadband** — it is a spike. That is what makes it
both dangerous (it is exactly the shape of a detection) and tractable (four
parameters describe it).

In [ ]:
hi = load('Fastbox_cube_cropped.npy')
occupied = np.unique(idxs)
occupied = occupied[occupied > 0]


def pk(cube):
    """Binned P(k), same convention and binning as the sampler's SCS step."""
    A = np.fft.fftn(cube - cube.mean(), norm='ortho').flatten()
    p = (A * np.conj(A)).real
    return np.array([p[idxs == b].mean() for b in occupied])


fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(sig_k, pk(hi), color=BLUE, marker='o', ms=5, label='simulated H I')
ax.plot(sig_k, pk(residual), color=ORANGE, marker='s', ms=5,
        label=f'ground-spill residual after a {N_MODES}-mode clean')
ax.axvline(k_hit, color=MUTED, lw=1.0, ls=(0, (2, 2)))
ax.annotate(f'{PERIOD:g} MHz ripple, $k$ = {k_hit:.3f}',
            xy=(k_hit, np.sqrt(pk(hi)[0] * pk(residual)[0])), xytext=(8, 0),
            textcoords='offset points', fontsize=8.5, color=INK2, va='center')
# Direct labels at the right ends, where the two curves are well separated.
for series, colour, name in ((pk(hi), BLUE, 'simulated H I'),
                             (pk(residual), ORANGE, 'spill residual')):
    ax.annotate(name, xy=(sig_k[-1], series[-1]), xytext=(7, 0),
                textcoords='offset points', fontsize=8.5, color=colour,
                va='center')
ax.set_xlim(sig_k[0] * 0.72, sig_k[-1] * 2.9)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$k$ [Mpc$^{-1}$]')
ax.set_ylabel(r'$P(k)$ [K$^2$]')
ax.set_title('Ripple residual against the signal it contaminates', loc='left')
ax.legend(loc='lower left')
bare(ax)
fig.tight_layout()

ratio = pk(residual) / pk(hi)
for i, (k, r) in enumerate(zip(sig_k, ratio)):
    print(f'bin {i}  k = {k:7.4f}   residual / H I = {r:8.2f}')

## 7. Polarisation leakage: the same story, told in $\lambda^2$

Polarised synchrotron is Faraday-rotated, so what leaks from Q/U into total
intensity oscillates as $\cos(2\chi_0 + 2\,\mathrm{RM}\,\lambda^2)$. The two
quadratures are carried separately again, which makes the polarisation angle
$\chi_0$ a linear parameter and leaves the rotation measure as the one fixed
nonlinear knob — structurally identical to the standing-wave period.

The measurement below is the one that matters, and it is discouraging in a
useful way. This band spans $\lambda^2 = 0.0859$–$0.0953\ \mathrm{m^2}$, a
range of only $0.0095$, so the number of cycles is
$\mathrm{RM}\times0.0095/\pi$. **Ordinary Galactic Faraday depths are tens of
rad m$^{-2}$** — which turns through a few hundredths of a cycle. Smooth,
absorbed whole, invisible.

Only $\mathrm{RM}\gtrsim500$ leaves anything at all. Widening the band is what
buys sensitivity to lower RM, which is a concrete argument for the
500-channel cut if leakage ever turns out to matter.

In [ ]:
rms_grid = np.logspace(np.log10(20), np.log10(3000), 140)


def absorbed_by_fg(templates, fg):
    """Power fraction of each template inside the span of `fg`, averaged."""
    t = templates / np.linalg.norm(templates, axis=1)[:, None]
    return float(np.mean(np.sum((fg @ t.T) ** 2, axis=0)))


fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(7.2, 5.8),
                               gridspec_kw={'height_ratios': [1.2, 1]})

for n_modes, colour in ((6, BLUE), (10, ORANGE), (20, AQUA)):
    fgb = legendre_basis(len(FREQS), n_modes)
    curve = [absorbed_by_fg(faraday_templates(FREQS, rm=r), fgb)
             for r in rms_grid]
    ax0.plot(rms_grid, curve, color=colour, label=f'{n_modes} modes')
    ax0.annotate(f'{n_modes} modes',
                 xy=(np.interp(0.5, curve[::-1], rms_grid[::-1]), 0.5),
                 xytext=(9, 16), textcoords='offset points', ha='left',
                 fontsize=8.5, color=colour)

ax0.axvspan(10, 100, color=GRID, alpha=0.7, lw=0)
ax0.annotate('typical Galactic RM\n(absorbed whole)', xy=(32, 0.42),
             fontsize=8.5, color=INK2, ha='center')
ax0.set_xscale('log')
ax0.set_ylim(-0.04, 1.1)
ax0.set_ylabel('fraction absorbed by the foreground')
ax0.set_title('Leakage is only identifiable at high Faraday depth', loc='left')
ax0.legend(loc='lower left', fontsize=8.5)
bare(ax0)

# The chirp: periodic in lambda^2, so the period in FREQUENCY drifts.
l2 = lambda_squared(FREQS)
ax1.plot(FREQS, np.cos(2 * RM_DEFAULT * l2), color=ORANGE)
zeros = FREQS[np.where(np.diff(np.sign(np.cos(2 * RM_DEFAULT * l2))))[0]]
for z in zeros:
    ax1.axvline(z, color=MUTED, lw=0.7, ls=(0, (2, 3)))
ax1.set_xlabel('frequency [MHz]')
ax1.set_ylabel(r'$\cos(2\,\mathrm{RM}\,\lambda^2)$')
ax1.set_title(f'RM = {RM_DEFAULT:g} rad m$^{{-2}}$: the zero crossings are not '
              'evenly spaced', loc='left')
bare(ax1)
fig.tight_layout()

# The drift is a LOCAL property, so measure it from the phase gradient at the
# two band edges. Zero-crossing gaps understate it, because each gap averages
# the period over a finite interval rather than sampling it at a point.
grad = np.gradient(2 * RM_DEFAULT * l2, FREQS)
period_lo, period_hi = 2 * np.pi / np.abs(grad[0]), 2 * np.pi / np.abs(grad[-1])
print(f'local period: {period_lo:.3f} MHz at the low edge, {period_hi:.3f} MHz '
      f'at the high edge  ({period_hi / period_lo:.3f}x)')
print(f'predicted (nu_hi/nu_lo)^3 = {(FREQS[-1] / FREQS[0]) ** 3:.3f}')
gaps = np.diff(zeros)
print(f'zero-crossing gaps run {gaps[0]:.3f} -> {gaps[-1]:.3f} MHz '
      f'({gaps[-1] / gaps[0]:.3f}x), lower because each gap is an average '
      f'over its own interval')
fg6 = legendre_basis(len(FREQS), 6)
for rm in (10, 100, 500, 1000, 2000):
    print(f'  RM {rm:5d}: {rm*(l2.max()-l2.min())/np.pi:5.2f} cycles, '
          f'FG absorbs {absorbed_by_fg(faraday_templates(FREQS, rm=rm), fg6):6.1%}')

## 8. 1/f: a basis that derives its own prior

1/f is a *stochastic process*, not a fixed template, so unlike the other two it
has no natural low-rank basis. What it does have is a covariance — and the
leading Karhunen–Loève modes of that covariance are exactly the directions
carrying most of its variance.

Truncating there gives a basis in the form this module already uses, and it
buys something the other two do not have: **the eigenvalues are the prior
variances.** For ground spill, $G$ has to come from instrument
characterisation, because nothing says one template should carry more
amplitude than another. Here the model supplies it.

The other difference is that the foreground degeneracy is handled *before the
fact*. A gain fluctuation that moves every channel together is constant in
frequency, so it is inside the foreground span to machine precision — the same
statement as smooth ground spill. Rather than discover that afterwards,
`onef_basis` deflates the foreground out of the frequency covariance before
taking the modes, so the retained modes are orthogonal to it by construction.

There is a caveat on the "derived prior" claim, and the numbers below make it.
Before deflation the eigenvalues fall $1.00, 1.00, 0.75, 0.75, 0.67, \ldots$
— in degenerate pairs, the sine and cosine of each Fourier mode, which is what
a stationary process on a periodic domain gives you. *After* deflation they
run $1.00$ down to only $0.84$. The foreground has taken the reddest part of
the process (18% of the variance in the leading modes), and what is left is
close to flat. So the prior is principled rather than guessed, but it is only
mildly informative — do not expect the mode ordering to do much work.

In [ ]:
fg6 = legendre_basis(len(FREQS), N_MODES)
basis_deflated, prior_deflated = onef_basis(FREQS, SHAPE, n_scan=3, n_spec=3,
                                            fg_basis=fg6)
basis_plain, _ = onef_basis(FREQS, SHAPE, n_scan=3, n_spec=3, fg_basis=None)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 3.4),
                         gridspec_kw={'width_ratios': [1, 1, 0.85]})

for i, row in enumerate(basis_deflated.spatial):
    axes[0].plot(row.reshape(SHAPE[0], SHAPE[1])[:, 0],
                 color=[BLUE, ORANGE, AQUA][i], label=f'mode {i}')
axes[0].set_xlabel('RA pixel (scan direction)')
axes[0].set_ylabel('unit-RMS amplitude')
axes[0].set_title('KL modes along the scan', loc='left')
axes[0].legend(fontsize=8.5)
bare(axes[0])

for i, row in enumerate(basis_deflated.spectral):
    axes[1].plot(FREQS, row, color=[BLUE, ORANGE, AQUA][i], label=f'mode {i}')
axes[1].set_xlabel('frequency [MHz]')
axes[1].set_title('KL modes across frequency, foreground deflated', loc='left')
axes[1].legend(fontsize=8.5)
bare(axes[1])

x = np.arange(prior_deflated.size)
axes[2].bar(x, np.sort(prior_deflated.ravel())[::-1], color=BLUE,
            edgecolor='none', width=0.68)
axes[2].set_xlabel('mode (ordered)')
axes[2].set_ylabel('relative prior variance')
axes[2].set_title(r'$G$ from the KL eigenvalues', loc='left')
axes[2].set_xticks(x)
bare(axes[2])
fig.tight_layout()

print('overlap of the spectral modes with the foreground basis:')
print(f'  deflated : {np.abs(fg6 @ basis_deflated.spectral.T).max():.2e}')
print(f'  plain    : {np.abs(fg6 @ basis_plain.spectral.T).max():.2e}')
print(f'\nprior variances (sum to 1): '
      + ', '.join(f'{v:.3f}' for v in np.sort(prior_deflated.ravel())[::-1]))

# How much flatter did deflation make the spectrum?
from imgibbs.systematics import onef_covariance, _kl_modes
_, plain_vals = _kl_modes(onef_covariance(SHAPE[2]), 8)
_, defl_vals = _kl_modes(onef_covariance(SHAPE[2]), 8, deflate=fg6)
print('frequency eigenvalues, relative to the largest:')
print('  no deflation : ' + ', '.join(f'{v:.3f}' for v in plain_vals / plain_vals[0]))
print('  deflated     : ' + ', '.join(f'{v:.3f}' for v in defl_vals / defl_vals[0]))
print(f'  the foreground took {1 - defl_vals.sum() / plain_vals.sum():.1%} of '
      f'the variance in these 8 modes -- the reddest part')

## 9. All three together — and a correction worth making

The useful comparison is not "how big is it" but **where does it land**. A
systematic that piles into one $k_\parallel$ mode can in principle be excised;
one that spreads cannot.

It is tempting to predict that ground spill is a spike (fixed period), leakage
is smeared (the period drifts), and 1/f is broadband. Two of those are right.
**The leakage prediction is wrong on this band, and the reason is worth
knowing.** The chirp is real — the local period drifts by 17% from one band
edge to the other — but the $k_\parallel$ modes here are spaced
$\Delta k_\parallel = 2\pi/L_z = 0.0247\ \mathrm{Mpc^{-1}}$, which at
$k \approx 0.074$ is a spacing of 33%. A 17% drift does not resolve. On 52 MHz
polarisation leakage is *indistinguishable* from a fixed-period ripple.

That is another concrete argument for the 500-channel cut: the chirp is the
thing that would let you tell leakage from ground spill without a model, and
it only becomes visible on a wider band.

And at the sampler's own 5 log-spaced bins, **all three land in bin 0** — so
the binned $P(k)$ cannot tell them apart at all. Separating them is what the
model is for.

In [ ]:
kpar = 2 * np.pi * np.fft.rfftfreq(SHAPE[2], d=grid.box_dims[2] / SHAPE[2])

gs_cube = groundspill_basis(FREQS, SHAPE, period=PERIOD).cube(
    np.array([[1.0, 0.0], [0.0, 0.0]]))
lk_cube = leakage_basis(FREQS, SHAPE, rm=RM_DEFAULT).cube(
    np.array([[1.0, 0.0], [0.0, 0.0], [0.0, 0.0]]))
onef_cube = basis_deflated.cube(np.sqrt(prior_deflated))


def kpar_spectrum(cube):
    """Power along the frequency axis only, averaged over pixels, normalised."""
    spec = np.abs(np.fft.rfft(cube / cube.std(), axis=2)) ** 2
    p = spec.reshape(-1, len(kpar)).mean(axis=0)
    return p / p.sum()


fig, ax = plt.subplots(figsize=(7.4, 4.2))
for cube, colour, name in ((gs_cube, BLUE, f'ground spill, {PERIOD:g} MHz'),
                           (lk_cube, ORANGE, f'leakage, RM = {RM_DEFAULT:g}'),
                           (onef_cube, AQUA, '1/f')):
    ax.plot(kpar[1:], kpar_spectrum(cube)[1:], color=colour, marker='o', ms=4,
            label=name)
for edge in sig_k[:2]:
    ax.axvline(edge, color=MUTED, lw=0.9, ls=(0, (2, 3)))
ax.annotate('sampler bin centres 0 and 1', xy=(sig_k[0], 2.5e-5),
            xytext=(8, 0), textcoords='offset points', fontsize=8.5, color=INK2)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylim(1e-5, 2)
ax.set_xlabel(r'$k_\parallel$ [Mpc$^{-1}$]')
ax.set_ylabel('fraction of the systematic power')
ax.set_title(r'Native $k_\parallel$ resolution, not the sampler bins', loc='left')
ax.legend(loc='upper right', fontsize=8.5)
bare(ax)
fig.tight_layout()

print(f'k_par spacing {kpar[1]:.4f} Mpc^-1 '
      f'= {kpar[1] / 0.0741:.0%} of the ripple wavenumber\n')
print('                 peak k_par   within +-10% of peak   modes for 90%')
for cube, name in ((gs_cube, 'ground spill'), (lk_cube, 'leakage'),
                   (onef_cube, '1/f')):
    p = kpar_spectrum(cube)
    peak = kpar[np.argmax(p)]
    near = p[(kpar > 0.9 * peak) & (kpar < 1.1 * peak)].sum()
    n90 = int(np.searchsorted(np.cumsum(np.sort(p)[::-1]), 0.90)) + 1
    print(f'  {name:14s}   {peak:8.4f}        {near:8.1%}            {n90:3d}')

print('\nIn the sampler\'s 5 bins, all three sit in bin 0:')
for cube, name in ((gs_cube, 'ground spill'), (lk_cube, 'leakage'),
                   (onef_cube, '1/f')):
    p = pk(cube / cube.std())
    print(f'  {name:14s} bin 0 holds {p[0] / p.sum():.4f} of the power')

## Where to go from here

The block is off by default everywhere — `sys_basis=None` leaves the published
three-block sampler numerically identical. To turn it on:

```bash
python scripts/run_gibbs.py 6 --groundspill --gs-period 17.5
```

On real data you cannot distinguish a systematic that was *removed* from one
that was never *there* — the same structural limitation `docs/STATUS.md`
records for the Gibbs-vs-PCA comparison. So the thing to run first is the
injection test, which builds a synthetic cube with a known answer and samples
it with identical seeds across arms:

```bash
python scripts/systematics_injection.py --systematic onef --arm clean
python scripts/systematics_injection.py --systematic onef --arm off
python scripts/systematics_injection.py --systematic onef --arm on
python scripts/groundspill_report.py --out outputs/onef_run
```

or, on a cluster, `sbatch scripts/submit_systematics.sh onef on`.

**The block can only ever recover the part the foreground does not already
take** — §2 and §7 are the ceiling on what any run can demonstrate. Both
scripts print it before they start.

### The one lesson that generalises

Everything smooth in frequency belongs to the foreground block already, and no
amount of spatial structure rescues it. That was true of the smooth spillover
envelope, it is true of the 1/f common mode, and it is true of polarisation
leakage at any Galactic Faraday depth. In each case the identifiable part is a
smaller, more structured thing than the systematic as a whole — and it is the
only part worth spending parameters on.

### Open

* **The nonlinear parameters are fixed, not sampled** — the ripple period and
  the rotation measure. Getting them wrong is the interesting failure mode and
  is untested; a search needs a nonlinear step.
* **`S` competes with the block.** `S` is estimated from a signal field whose
  power has already been distorted, and the next draw amplifies the
  distortion — it inflates in a contaminated bin and deflates in a
  foreground-degenerate one. See `docs/STATUS.md`.
* **A multiplicative systematic does not fit this block at all.** A gain error
  needs linearisation or its own Metropolis step.
* **There is no beam in the model**, and that is the next thing to add. It
  matters here more than it looks: the beam acts on the signal and the
  foreground but *not* on the systematics block, so the block is unaffected —
  but the beam is **chromatic** (FWHM 1.60° to 1.52° across this band), and a
  chromatic beam turns spatial structure into spectral structure. That breaks
  the assumption every number in this notebook rests on, that the foreground
  is smooth in frequency in each pixel. **The absorbed-fraction tables above
  would have to be re-measured.** See `docs/STATUS.md`, "No beam in the
  MODEL".